In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("Movies_csv.csv", low_memory=False)

df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
 df = df.drop_duplicates().reset_index(drop = True)

In [ ]:
df = df[['title' , 'overview' , 'genres' , 'tagline' , 'vote_average' , 'popularity']]

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset = ['title'])

In [ ]:
df['overview'] = df['overview'].fillna('')

In [ ]:
df.iloc[0]['genres']

In [ ]:
import ast

In [ ]:
df.head()

In [ ]:
df['tagline'] = df['tagline'].fillna('')

In [ ]:
df.isnull().sum()

In [ ]:
df['tags'] = df['overview'] + " " + df['genres'] + " " + df['tagline']

In [ ]:
df.head()

## NLP Preprocessing

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
!pip install nltk

In [ ]:
import nltk
print(nltk.__version__)

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [ ]:
df['tags'] = df['tags'].apply(preprocess_text)

In [ ]:
df = df.reset_index(drop = True)

In [ ]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

## Convert Words to Numerical Vectors

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(max_features = 50000 , ngram_range = (1,2) , stop_words = 'english')

In [ ]:
tfidf_matrix = tfidf.fit_transform(df['tags'])
tfidf_matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend(title , n = 10):
    if title not in indices:
        return ['Movie not found']

    idx = indices[title]
    sim_score = cosine_similarity(tfidf_matrix[idx] , tfidf_matrix).flatten()
    similar_idx = sim_score.argsort()[::-1][1:n+1]
    return df['title'].iloc[similar_idx]

In [ ]:
recommend('Avatar' , 5)

In [ ]:
import pickle
pickle.dump(tfidf_matrix , open('tfidf_matrix.pkl' , 'wb'))
pickle.dump(indices , open('indices.pkl' , 'wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf , open('tfidf.pkl' , 'wb'))